# Illustrated NN Building Blocks

`affon:nn` modules are callable blocks with parameters and mode. This notebook explains the beginner mental model for common blocks.

How to read each block:

1. Ask what problem the block solves.
2. Predict the input and output shape.
3. Run the cell.
4. Read which axes were preserved, transformed, normalized, or treated as time.


In [ ]:
import { tensor } from 'affon:compute'
import nn from 'affon:nn'

function show(label: string, value: { shape: readonly number[], to_array(): unknown }) {
  console.log(label)
  console.log('shape:', value.shape)
  console.log(JSON.stringify(value.to_array()))
}


## Linear

$$
y = xW + b
$$

For high-rank input, leading dimensions are parallel.

```txt
[batch, time, in_features] -> Linear(in, out) -> [batch, time, out_features]
```

<svg viewBox="0 0 760 190" width="760" height="190" xmlns="http://www.w3.org/2000/svg">
  <style>.txt{font:14px sans-serif;fill:#0f172a}.mono{font:15px monospace}.p{fill:#dbeafe;stroke:#60a5fa}.o{fill:#dcfce7;stroke:#4ade80}.q{fill:#ffedd5;stroke:#fb923c}</style>
  <text class="txt" x="24" y="28">For every parallel position, project one feature vector.</text>
  <rect class="p" x="70" y="70" width="80" height="40" rx="4"/><text class="mono" x="91" y="95">batch</text>
  <rect class="p" x="155" y="70" width="70" height="40" rx="4"/><text class="mono" x="176" y="95">time</text>
  <rect class="o" x="230" y="70" width="110" height="40" rx="4"/><text class="mono" x="248" y="95">in_features</text>
  <text class="mono" x="375" y="95">Linear</text>
  <rect class="p" x="470" y="70" width="80" height="40" rx="4"/><text class="mono" x="491" y="95">batch</text>
  <rect class="p" x="555" y="70" width="70" height="40" rx="4"/><text class="mono" x="576" y="95">time</text>
  <rect class="q" x="630" y="70" width="120" height="40" rx="4"/><text class="mono" x="648" y="95">out_features</text>
</svg>


### Before You Run

A Linear layer changes only the feature dimension. Predict the output shape for input `[2, 3]` through `Linear(3, 2)`.

### Read The Output

The exact numbers are random because the layer is initialized randomly. The important beginner fact is the shape: each length-3 row becomes a length-2 row.


In [ ]:
const linear = nn.Linear(3, 2)
show('linear(tensor([[1,2,3],[4,5,6]]))', linear(tensor([[1, 2, 3], [4, 5, 6]])))


## Embedding

An embedding layer is a lookup table.

```txt
token ids: [tokens]
table:     [vocab, embedding_dim]
output:    [tokens, embedding_dim]
```


### Before You Run

For token ids shaped `[2]` and an embedding dim of `3`, predict the output shape.

### Read The Output

Each integer id selects one row from the embedding table. The id axis stays, and a new embedding feature axis appears.


In [ ]:
const embedding = nn.Embedding(5, 3)
show('embedding([1, 3])', embedding(tensor([1, 3], { dtype: 'i64' })))


## Sequential And Module List

`Sequential` pipes one block into the next.

```txt
input -> Linear -> activation -> Linear -> output
```

`module_list` owns repeated blocks so their parameters are included in the module state.


In [ ]:
const model = nn.Sequential(nn.Linear(3, 4), nn.Linear(4, 2))
show('Sequential output', model(tensor([[1, 2, 3]])))
const layers = nn.module_list([nn.Linear(3, 3), nn.Linear(3, 3)])
console.log('module_list length:', layers.length)
console.log('module_list parameters:', layers.parameters.length)


## Dropout

Dropout randomly zeros activations during training. In eval mode, it passes values through.

```txt
train: some values are zeroed
eval:  values pass through
```


### Before You Run

In eval mode, predict whether the values will change. In train mode, some values may be randomly zeroed.


In [ ]:
const dropout = nn.Dropout(0.5)
dropout.mode('eval')
show('dropout eval', dropout(tensor([[1, 2], [3, 4]])))


## LayerNorm And BatchNorm

Normalization blocks rescale activations so training is easier.

- `LayerNorm`: normalize within each example over the feature axis.
- `BatchNorm`: normalize each feature using batch statistics.


### Before You Run

LayerNorm normalizes within each row over features. BatchNorm normalizes each feature using batch statistics. Predict which axis each one treats as the feature axis in the example.


In [ ]:
const layerNorm = nn.LayerNorm(3)
show('LayerNorm over last dim', layerNorm(tensor([[1, 2, 3], [4, 5, 6]])))
const batchNorm = nn.BatchNorm(3)
show('BatchNorm over features', batchNorm(tensor([[1, 2, 3], [4, 5, 6]])))


## Recurrent Blocks

`SimpleRNN`, `RNN`, and `LSTM` process sequences by carrying state through time.

```txt
x_t + previous hidden -> new hidden
```

`LSTM` also carries a cell state, which acts like a longer-term memory path.


### Before You Run

For batch-first RNN input `[batch=2, time=2, input=3]` with hidden size `4`, predict the output shape.

### Read The Output

The input feature axis becomes hidden size. Batch and time remain visible because the recurrent block emits a hidden vector for each time step.


In [ ]:
const rnn = nn.RNN(3, 4, { batch_first: true })
show('RNN batch_first output: [batch=2, time=2, hidden=4]', rnn(tensor([[[1, 2, 3], [4, 5, 6]], [[1, 1, 1], [2, 2, 2]]])))


## Masks, Positions, Init, Diagnostics

- `causal_mask` and `apply_causal_mask` prevent attention from seeing future positions.
- `position_ids` and `sinusoidal_encoding` add position information.
- `nn.init.*` helpers initialize tensors.
- `nn.diagnostics.*` can check values while a model runs.


In [ ]:
show('causal_mask(4)', nn.causal_mask(4))
show('position_ids(4)', nn.position_ids(4))
show('sinusoidal_encoding(4, 6)', nn.sinusoidal_encoding(4, 6))
